# PS S6E6 — Stellar Classification: EDA

**Goal:** Understand the data before touching any model.  
**Metric:** Balanced accuracy (GALAXY / STAR / QSO)  
**Key questions to answer:**
- Is there class imbalance? (affects baseline strategy)
- Which features separate classes best?
- Are there nulls or data quality issues?
- How do the photometric color indices look per class?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='darkgrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
train = pd.read_csv('train.csv', index_col='id')
test  = pd.read_csv('test.csv',  index_col='id')

print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head()

## 2. Schema & Data Quality

In [ ]:
train.info()

In [ ]:
# Null counts — train and test side by side
nulls = pd.DataFrame({
    'train_nulls': train.isnull().sum(),
    'train_%':     train.isnull().mean().mul(100).round(2),
    'test_nulls':  test.isnull().sum(),
    'test_%':      test.isnull().mean().mul(100).round(2),
})
nulls[nulls[['train_nulls', 'test_nulls']].sum(axis=1) > 0] if nulls[['train_nulls', 'test_nulls']].sum().sum() > 0 else print('No nulls found.')

In [ ]:
# Duplicate rows
print(f'Train duplicates: {train.duplicated().sum()}')
print(f'Test  duplicates: {test.duplicated().sum()}')

In [ ]:
train.describe()

## 3. Target Distribution

In [ ]:
class_counts = train['class'].value_counts()
class_pct    = train['class'].value_counts(normalize=True).mul(100).round(2)

print(pd.DataFrame({'count': class_counts, '%': class_pct}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
class_counts.plot.bar(ax=axes[0], title='Class Counts', rot=0)
axes[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Share')
plt.tight_layout()

## 4. Categorical Features

In [ ]:
cat_cols = ['spectral_type', 'galaxy_population']

for col in cat_cols:
    print(f'\n--- {col} ---')
    print(train[col].value_counts())

In [ ]:
# How do categoricals interact with the target?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, cat_cols):
    ct = pd.crosstab(train[col], train['class'], normalize='index').mul(100)
    ct.plot.bar(ax=ax, title=f'{col} vs class (row %)', rot=45)
    ax.set_ylabel('% within category')
    ax.legend(title='class')

plt.tight_layout()

## 5. Numerical Feature Distributions

In [ ]:
num_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
classes  = train['class'].unique()

fig, axes = plt.subplots(len(num_cols), 1, figsize=(12, 4 * len(num_cols)))

for ax, col in zip(axes, num_cols):
    for cls in classes:
        subset = train.loc[train['class'] == cls, col]
        subset.plot.kde(ax=ax, label=cls, linewidth=1.5)
    ax.set_title(col)
    ax.legend()
    ax.set_xlabel('')

plt.tight_layout()

## 6. Redshift Deep Dive

Redshift is the single strongest discriminator — QSOs typically have very high redshift.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram
for cls in classes:
    subset = train.loc[train['class'] == cls, 'redshift']
    axes[0].hist(subset, bins=100, alpha=0.5, label=cls, density=True)
axes[0].set_title('Redshift distribution (linear)')
axes[0].legend()

# Box plot
train.boxplot(column='redshift', by='class', ax=axes[1])
axes[1].set_title('Redshift by class')
plt.suptitle('')
plt.tight_layout()

In [ ]:
# Redshift summary stats per class
train.groupby('class')['redshift'].describe().round(4)

## 7. Photometric Color Indices

In astronomy, **color indices** (magnitude differences between filter bands) are the standard features for object classification.  
These differences cancel out distance-dependent brightness, leaving only spectral shape.

In [ ]:
def add_colors(df):
    df = df.copy()
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']
    return df

train_c = add_colors(train)
color_cols = ['u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'g_z']

print('Color index stats:')
train_c[color_cols].describe().round(3)

In [ ]:
fig, axes = plt.subplots(len(color_cols), 1, figsize=(12, 3.5 * len(color_cols)))

for ax, col in zip(axes, color_cols):
    for cls in classes:
        subset = train_c.loc[train_c['class'] == cls, col]
        subset.plot.kde(ax=ax, label=cls, linewidth=1.5)
    ax.set_title(f'Color index: {col}')
    ax.legend()

plt.tight_layout()

## 8. Color-Color Diagrams

The canonical astronomy diagnostic: plot two color indices against each other. Stars, galaxies, and QSOs occupy distinct loci.

In [ ]:
sample = train_c.groupby('class', group_keys=False).sample(n=3000, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs = [('u_g', 'g_r'), ('g_r', 'r_i'), ('r_i', 'i_z')]
palette = {'GALAXY': '#4c72b0', 'STAR': '#dd8452', 'QSO': '#55a868'}

for ax, (x, y) in zip(axes, pairs):
    for cls, color in palette.items():
        mask = sample['class'] == cls
        ax.scatter(sample.loc[mask, x], sample.loc[mask, y],
                   s=4, alpha=0.4, label=cls, color=color)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f'{x} vs {y}')
    ax.legend(markerscale=3)

plt.suptitle('Color-Color Diagrams (3k sample per class)', y=1.01)
plt.tight_layout()

## 9. Correlation Matrix

In [ ]:
all_num = num_cols + color_cols
corr = train_c[all_num].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()

## 10. Train vs Test Distribution Check

Verify there's no significant distribution shift between train and test.

In [ ]:
# KS test: small p-value = significant shift
ks_results = []
for col in num_cols:
    stat, p = stats.ks_2samp(train[col].dropna(), test[col].dropna())
    ks_results.append({'feature': col, 'ks_stat': stat, 'p_value': p})

ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)
print(ks_df.to_string(index=False))

In [ ]:
# Visual: overlaid density for each numerical feature
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    train[col].plot.kde(ax=ax, label='train', linewidth=1.5)
    test[col].plot.kde(ax=ax, label='test',  linewidth=1.5, linestyle='--')
    ax.set_title(col)
    ax.legend()

plt.suptitle('Train vs Test distributions', y=1.01)
plt.tight_layout()

## 11. EDA Summary

Fill this in after running the notebook:

| Finding | Impact |
|---|---|
| Class balance: ... | ... |
| Nulls: ... | ... |
| Strongest feature(s): ... | ... |
| Color indices useful: yes/no | ... |
| Train/test shift: ... | ... |
| Anything surprising: ... | ... |

**Features to prioritise in Week 2:** ...